# 01 — Molecular Structure & Basis Sets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/notebooks/01_molecular_structure_and_basis_sets.ipynb)

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
- Explain the difference between Slater-type (STO) and Gaussian-type (GTO) basis functions
- Compare basis set families and understand accuracy/cost trade-offs
- Perform basis set convergence studies for total energies
- Understand contracted vs primitive basis functions
- Understand effective core potentials (ECPs) and when to use them
- Visualize atomic orbital radial probability distributions

## 1. Theory: Basis Functions in Quantum Chemistry

### 1.1 Slater-Type Orbitals (STOs)

The exact hydrogen-like atomic orbitals are Slater-type orbitals (STOs):

$$\chi_{nlm}(r,\theta,\phi) = N r^{n-1} e^{-\zeta r} Y_l^m(\theta,\phi)$$

where:
- $N$ = normalization constant
- $n$ = principal quantum number
- $\zeta$ (zeta) = orbital exponent controlling size
- $Y_l^m$ = spherical harmonic

**Advantages:** Correct cusp at nucleus, correct exponential decay
**Disadvantage:** Multi-center two-electron integrals are very expensive (no analytic formula)

### 1.2 Gaussian-Type Orbitals (GTOs)

In practice, we use Gaussian-type orbitals (GTOs):

$$g_{lmn}(\mathbf{r}; \alpha) = N x^l y^m z^n e^{-\alpha r^2}$$

where $\alpha$ is the Gaussian exponent and $l+m+n$ determines the angular momentum.

**Advantage:** Products of Gaussians are Gaussians → all integrals are analytic!
**Disadvantage:** Incorrect behavior at nucleus (no cusp) and at long range (decays too fast)

### 1.3 Contracted Basis Sets

To capture STO-like behavior, we use **contracted** basis functions:
$$\chi_\mu = \sum_k d_{k\mu} g_k$$

A **contraction** combines multiple primitive Gaussians ($g_k$) with fixed coefficients ($d_{k\mu}$) to form a single basis function. For example, STO-3G uses 3 Gaussians to approximate each STO.

### 1.4 Basis Set Families

| Family | Strategy | Example |
|--------|----------|---------|
| **Pople** | Add shells systematically | 6-31G, 6-31G\*, 6-311G\*\* |
| **Dunning** (cc-pVxZ) | Converges to CBS limit | cc-pVDZ, cc-pVTZ, cc-pVQZ |
| **Ahlrichs** (def2) | Balanced accuracy/cost | def2-SVP, def2-TZVP, def2-QZVP |

**Notation:**
- **DZ** (double-ζ), **TZ** (triple-ζ), **QZ** (quadruple-ζ): number of basis functions per orbital
- **\*** or **P**: adds polarization functions (e.g., d functions on C, N, O)
- **aug-**: adds diffuse functions (needed for anions, excited states, weak interactions)

In [ ]:
# =============================================================================
# Ch121a: Quantum Chemistry & DFT — Notebook 01: Molecular Structure & Basis Sets
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyscf import gto, scf

In [ ]:
%%time
# ------------------------------------------------------------------
# Basis set convergence study for water (H2O)
# ------------------------------------------------------------------
# We compare HF energies across a range of basis sets to see
# how the energy converges toward the complete basis set (CBS) limit.

mol_template = """
O   0.000000   0.000000   0.117176
H   0.000000   0.757001  -0.468704
H   0.000000  -0.757001  -0.468704
"""

basis_sets = ['STO-3G', '3-21G', '6-31G*', 'cc-pVDZ', 'cc-pVTZ', 'def2-SVP', 'def2-TZVP']

results = []
for basis in basis_sets:
    mol = gto.Mole()
    mol.atom = mol_template
    mol.basis = basis
    mol.verbose = 0
    mol.build()
    mf = scf.RHF(mol)
    mf.verbose = 0
    e = mf.kernel()
    n_bas = mol.nao_nr()
    results.append({'Basis Set': basis, 'N_basis': n_bas,
                    'E_HF (Ha)': round(e, 8), 'E_HF (eV)': round(e*27.2114, 5)})
    print(f"  {basis:12s}  N={n_bas:3d}  E = {e:.8f} Ha")

df = pd.DataFrame(results)
print("\n")
print(df.to_string(index=False))
print(f"\ncc-pVTZ - STO-3G energy difference: {(results[4]['E_HF (Ha)']-results[0]['E_HF (Ha)'])*627.509:.2f} kcal/mol")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

energies = [r['E_HF (Ha)'] for r in results]
n_basis = [r['N_basis'] for r in results]
labels = [r['Basis Set'] for r in results]

# Panel 1: Energy vs basis set
ax1.plot(range(len(basis_sets)), energies, 'o-', color='steelblue',
         linewidth=2, markersize=8)
ax1.axhline(y=energies[-1], color='coral', linestyle='--', alpha=0.7, label='def2-TZVP (reference)')
ax1.set_xticks(range(len(basis_sets)))
ax1.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax1.set_xlabel('Basis Set')
ax1.set_ylabel('HF Energy (Hartree)')
ax1.set_title('Basis Set Convergence of HF Energy\nfor H₂O')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Panel 2: Number of basis functions vs basis set
colors = ['steelblue' if 'STO' in b or '3-21' in b or '6-31G*' in b else
          'seagreen' if 'cc-p' in b else 'coral' for b in labels]
bars = ax2.bar(range(len(basis_sets)), n_basis, color=colors, edgecolor='white', linewidth=0.5)
ax2.set_xticks(range(len(basis_sets)))
ax2.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax2.set_xlabel('Basis Set')
ax2.set_ylabel('Number of Basis Functions')
ax2.set_title('Basis Set Size for H₂O')
ax2.grid(True, alpha=0.3, axis='y')
for bar, n in zip(bars, n_basis):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(n), ha='center', va='bottom', fontsize=9)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', label='Pople'),
                   Patch(facecolor='seagreen', label='Dunning (cc-pVxZ)'),
                   Patch(facecolor='coral', label='Ahlrichs (def2)')]
ax2.legend(handles=legend_elements, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Effective Core Potentials (ECPs)
# ------------------------------------------------------------------
# For heavy elements (3d, 4d, 5d transition metals, lanthanides),
# the core electrons are chemically inert and their relativistic
# effects can be captured via an ECP (pseudopotential).

from pyscf import gto

# Demonstrate ECP for Iron (Fe, Z=26)
mol_Fe = gto.Mole()
mol_Fe.atom = 'Fe 0 0 0'
mol_Fe.basis = 'def2-SVP'
mol_Fe.ecp = 'def2-SVP'     # Use the def2-SVP ECP for Fe
mol_Fe.verbose = 0
mol_Fe.build()

print("=" * 55)
print("  Effective Core Potential (ECP) Demonstration: Fe")
print("=" * 55)
print(f"  Element:                        Fe (Z = 26)")
print(f"  ECP:                            def2-SVP")
print(f"  Core electrons replaced by ECP: {mol_Fe.atom_nelec_core(0)}")
print(f"  Valence electrons treated:      {mol_Fe.nelectron}")
print(f"  Basis functions (valence only): {mol_Fe.nao_nr()}")

# Compare with all-electron def2-SVP (no ECP)
mol_Fe_ae = gto.Mole()
mol_Fe_ae.atom = 'Fe 0 0 0'
mol_Fe_ae.basis = 'def2-SVP'
mol_Fe_ae.verbose = 0
mol_Fe_ae.build()
print(f"\n  All-electron comparison:")
print(f"  All electrons:                  {mol_Fe_ae.nelectron}")
print(f"  Basis functions (all-electron): {mol_Fe_ae.nao_nr()}")

print("\nWhen to use ECPs:")
print("  • Elements beyond Kr (Z > 36): relativistic effects important")
print("  • 3d metals: saves ~10 core electrons per atom")
print("  • 4d/5d metals: saves ~28/60 core electrons per atom")
print("  • Lanthanides/Actinides: ECP is essentially mandatory")

In [ ]:
# ------------------------------------------------------------------
# Radial Probability Distributions of Hydrogen-like Orbitals
# ------------------------------------------------------------------
# P(r) = r^2 |R_nl(r)|^2  (probability per unit radius)

import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import factorial, genlaguerre

def hydrogen_radial(n, l, r, Z=1):
    """Normalized radial wavefunction R_nl(r) for hydrogen-like atom."""
    rho = 2 * Z * r / n
    # Normalization constant
    norm = np.sqrt((2*Z/n)**3 * factorial(n-l-1) /
                   (2*n * factorial(n+l)**3))
    # Associated Laguerre polynomial
    L = genlaguerre(n-l-1, 2*l+1)(rho)
    return norm * np.exp(-rho/2) * rho**l * L

r = np.linspace(0.001, 30, 2000)  # Bohr radii

orbitals = [
    (1, 0, '1s', 'steelblue',   '-'),
    (2, 0, '2s', 'coral',        '-'),
    (2, 1, '2p', 'seagreen',    '-'),
    (3, 0, '3s', 'mediumpurple', '--'),
    (3, 1, '3p', '#E67E22',      '--'),
    (3, 2, '3d', '#E74C3C',      '--'),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left plot: all orbitals together
ax = axes[0]
for n, l, label, color, ls in orbitals:
    R = hydrogen_radial(n, l, r)
    P = r**2 * R**2
    ax.plot(r, P, label=f'${label}$', color=color, linestyle=ls, linewidth=2)
ax.set_xlim(0, 25)
ax.set_ylim(bottom=0)
ax.set_xlabel('Radius $r$ (Bohr)', fontsize=12)
ax.set_ylabel('Radial Probability $P(r) = r^2|R_{nl}|^2$', fontsize=11)
ax.set_title('Hydrogen-like Radial Probability Distributions', fontsize=12)
ax.legend(fontsize=11, ncol=2)
ax.grid(True, alpha=0.3)

# Right plot: n=1,2,3 s orbitals only (compare size)
ax2 = axes[1]
for n, l, label, color, ls in orbitals[:3]:
    R = hydrogen_radial(n, l, r)
    P = r**2 * R**2
    # Mark the most probable radius
    r_mp = r[np.argmax(P)]
    ax2.plot(r, P, label=f'${label}$ ($r_{{mp}}$={r_mp:.1f} $a_0$)',
             color=color, linewidth=2.5)
    ax2.axvline(x=r_mp, color=color, linestyle=':', alpha=0.6)
ax2.set_xlim(0, 15)
ax2.set_ylim(bottom=0)
ax2.set_xlabel('Radius $r$ (Bohr)', fontsize=12)
ax2.set_ylabel('Radial Probability $P(r)$', fontsize=11)
ax2.set_title('1s, 2s, 2p: Most Probable Radii', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("\nNote: 1 Bohr = 0.529 Å")
print("Most probable radius for 1s: 1 Bohr (= Bohr radius a₀)")

## 2. Basis Set Reference Table

| Basis Set | Family | # Fcns (H₂O) | Cost | Accuracy | Best Use Case |
|-----------|--------|:------------:|:----:|:--------:|---------------|
| STO-3G | Pople minimal | 7 | ★ | ★ | Quick tests, teaching |
| 3-21G | Pople split-val | 13 | ★★ | ★★ | Qualitative trends |
| 6-31G\* | Pople + polariz | 19 | ★★ | ★★★ | Organic chemistry |
| 6-311G\*\* | Pople triple-ζ | 30 | ★★★ | ★★★★ | Improved properties |
| cc-pVDZ | Dunning DZ | 24 | ★★ | ★★★ | Correlated calculations |
| cc-pVTZ | Dunning TZ | 58 | ★★★★ | ★★★★ | High accuracy |
| aug-cc-pVDZ | Dunning aug-DZ | 41 | ★★★ | ★★★★ | Anions, polarizability |
| def2-SVP | Ahlrichs SVP | 24 | ★★ | ★★★ | DFT, geometry opt |
| def2-TZVP | Ahlrichs TZV | 58 | ★★★★ | ★★★★★ | Production DFT |

**Rules of thumb:**
- Start with **def2-SVP** for geometry optimizations
- Use **def2-TZVP** for final energies and properties
- Use **cc-pVTZ** or higher for correlated calculations (MP2, CCSD)
- Add **aug-** prefix for anions, Rydberg states, polarizabilities, NMR
- Use **ECPs** (def2-SVP/TZVP) for transition metals and heavy elements

## 🔬 Research Connection

Basis set choice is critical in published computational chemistry papers:

- **Protein active sites**: def2-SVP is often used for QM/MM calculations on enzyme
  active sites (50-200 atoms in the QM region) due to its cost/accuracy balance.
- **CBS extrapolation**: High-accuracy thermochemistry (e.g., W4 theory, HEAT protocol)
  uses cc-pVDZ through cc-pV5Z with extrapolation to the complete basis set limit.
- **ORCA defaults**: ORCA uses def2-SVP as the default basis for geometry optimization
  and def2-TZVP for single-point energies — matching our course workflow.

**Key resource:** The EMSL Basis Set Exchange (https://www.basissetexchange.org/)
provides basis sets in formats for all major QC programs.

## 📋 Summary

**Key concepts:**
- **STOs** are physically correct but computationally expensive; **GTOs** are analytically tractable and used in all modern codes
- **Contracted** basis functions = fixed combinations of primitives; reduce computational cost
- **Convergence hierarchy**: STO-3G < 3-21G < 6-31G\* < cc-pVDZ < cc-pVTZ ← HF limit
- **Polarization functions** (\*) are essential for geometry and properties
- **Diffuse functions** (aug-) needed for anions, excited states, and weak interactions
- **ECPs** replace chemically inert core electrons in heavy elements with a potential

**Practical recommendation for this course:**
- Geometry optimization: **B3LYP/def2-SVP**
- Single-point energies: **B3LYP/def2-TZVP** or **PBE0/def2-TZVP**
- Heavy elements (3d metals): **def2-SVP** with ECP

## 📝 Exercises

1. **STO-3G vs cc-pVTZ**: Run HF/STO-3G and HF/cc-pVTZ on CO₂. 
   What is the C=O bond length predicted by each? Compare with experimental 1.162 Å.
   (Hint: use `mol.atom = 'C 0 0 0; O 0 0 1.162; O 0 0 -1.162'`)

2. **Convergence plot**: Extend the basis set convergence study to include 
   `aug-cc-pVDZ` and `aug-cc-pVTZ`. How do the energies compare to the cc- series?

3. **ECP for copper**: Create a PySCF Mole object for Cu using def2-SVP with ECP.
   How many core electrons are replaced? How does this compare to Fe?

4. **Radial probability**: Modify the radial probability plot to show the 4s and 3d 
   orbitals. Why are these two orbitals similar in energy for transition metals?

5. **Basis set superposition error (BSSE)**: Look up the counterpoise correction method.
   Write pseudocode describing how you would estimate BSSE for the water dimer.